# Smoke Test ClickHouse

Notebook này kiểm tra kết nối ClickHouse local bằng `clickhouse-connect`, sau đó chạy một query kiểm tra và một thao tác tạo bảng tối thiểu.

In [4]:
from datetime import datetime
import os

import clickhouse_connect

CLICKHOUSE_HOST = os.getenv("CLICKHOUSE_HOST", "localhost")
CLICKHOUSE_PORT = int(os.getenv("CLICKHOUSE_PORT", "8123"))
CLICKHOUSE_USER = os.getenv("CLICKHOUSE_USER", "admin")
CLICKHOUSE_PASSWORD = os.getenv("CLICKHOUSE_PASSWORD", "admin123")
CLICKHOUSE_DB = os.getenv("CLICKHOUSE_DB", "lakehouse")

client = clickhouse_connect.get_client(
    host=CLICKHOUSE_HOST,
    port=CLICKHOUSE_PORT,
    username=CLICKHOUSE_USER,
    password=CLICKHOUSE_PASSWORD,
    database=CLICKHOUSE_DB,
)

print(f"Connected to ClickHouse at {CLICKHOUSE_HOST}:{CLICKHOUSE_PORT} using database '{CLICKHOUSE_DB}'")

Connected to ClickHouse at localhost:8123 using database 'lakehouse'


## 1. Kiểm tra phiên bản và database hiện tại

Cell này xác nhận client có thể gửi query và nhận phản hồi từ server.

In [5]:
result = client.query(
    "SELECT version() AS version, currentDatabase() AS current_database, now() AS server_time"
)

print(result.result_rows)

[('24.3.18.7', 'lakehouse', datetime.datetime(2026, 6, 14, 7, 59, 47))]


## 2. Tạo bảng smoke test

Bảng này chỉ dùng để xác nhận DDL và INSERT hoạt động bình thường.

In [7]:
from datetime import datetime, timezone

table_name = "smoke_test_clickhouse"

client.command(f"DROP TABLE IF EXISTS {CLICKHOUSE_DB}.{table_name}")
client.command(
    f"""
    CREATE TABLE {CLICKHOUSE_DB}.{table_name} (
        id UInt32,
        message String,
        created_at DateTime
    )
    ENGINE = MergeTree
    ORDER BY id
    """
)

client.insert(
    f"{CLICKHOUSE_DB}.{table_name}",
    [[1, "clickhouse smoke test ok", datetime.now(timezone.utc)]],
    column_names=["id", "message", "created_at"],
)

rows = client.query(f"SELECT id, message, created_at FROM {CLICKHOUSE_DB}.{table_name} ORDER BY id").result_rows
print(rows)

[(1, 'clickhouse smoke test ok', datetime.datetime(2026, 6, 14, 7, 59, 59))]


## 3. Kết luận

Nếu notebook chạy qua hết các cell trên, ClickHouse đã sẵn sàng cho bước load Gold.